# Primetrade.ai Assignment Analysis

This notebook reproduces the full sentiment-versus-performance workflow for the Hyperliquid trader dataset and the Bitcoin Fear & Greed index.

## Executive takeaway

- Fear days are more active than Greed days: traders place more trades and use larger average size.
- Mean PnL is higher on Fear days, but median PnL is higher on Greed days, suggesting Fear creates bigger upside outliers while Greed is steadier.
- Segment behavior matters: high-size traders perform best in Fear, while low-size traders do better in Greed.
- A lightweight logistic model shows modest predictive signal for next-day positive profitability.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd
from primetrade_analysis.pipeline import run_full_analysis

outputs = run_full_analysis(output_root=PROJECT_ROOT / "outputs")
sentiment = outputs["sentiment"]
trades = outputs["trades"]
daily_account = outputs["daily_account"]
account_level = outputs["account_level"]
merged = outputs["merged"]


ModuleNotFoundError: No module named 'matplotlib'

## Data Quality Overview

In [ ]:
quality_overview = outputs["quality_tables"]["quality_overview"]
sentiment_missing = outputs["quality_tables"]["sentiment_missing"]
trade_missing = outputs["quality_tables"]["trade_missing"]
overlap_window = outputs["quality_tables"]["overlap_window"]

display(quality_overview)
display(overlap_window)
display(sentiment_missing.head(10))
display(trade_missing.head(10))


## Fear vs Greed Performance

In [ ]:
display(outputs["sentiment_summary_5class"])
display(outputs["sentiment_summary_binary"])


## Segment-Level Behavior

In [ ]:
display(outputs["size_segment_summary"])
display(outputs["frequency_segment_summary"])
display(outputs["consistency_segment_summary"])


## Suggested Insights

In [ ]:
top_lines = [
    f"Merged account-day rows: {len(merged):,}",
    f"Accounts covered: {account_level['account'].nunique():,}",
    f"Trade date range: {merged['date'].min().date()} to {merged['date'].max().date()}",
]
for line in top_lines:
    print(line)

display(outputs["key_insights"])
display(outputs["strategy_rules"])


## Submission-Ready Findings

1. **Fear days trigger more aggressive participation.** Average trade count rises from about `76.9` on Greed days to `105.4` on Fear days, and average trade size rises from about `$5.95k` to `$8.53k`.

2. **PnL shape changes by regime.** Mean daily closed PnL is higher on Fear days (`~5.19k` vs `~4.14k`), but the median account-day PnL is higher on Greed days (`~265` vs `~123`).

3. **Directional bias rotates with sentiment.** Fear days are more long-leaning, while Greed days become slightly short-leaning.

4. **Segment-aware rules are stronger than one-size-fits-all rules.** High-size traders are strongest on Fear days, but low-size traders do better on Greed days.

## Exported Figures

The pipeline saves figures into `outputs/figures/` for direct inclusion in the README or submission deck.

In [ ]:
figures_dir = PROJECT_ROOT / 'outputs' / 'figures'
sorted(path.name for path in figures_dir.glob('*.png'))


## Bonus Predictive Baseline

A simple logistic regression is included as an optional bonus. It predicts whether the next account-day for a trader will have positive realized PnL using sentiment, behavior, and segment features.

In [ ]:
display(outputs['model_metrics'])
